# Fashion-MNIST: Feature Exploration & HOG Visual Analysis

**Kartik Mehra · Milestone 2 · Team T112**  
*Project 11: Inventory Image Tagging via Handcrafted Features and a Small CNN*

This notebook explores the two visual feature representations evaluated in Phase 1 (Applied Machine Learning Baselines):
1. **Flattened Raw Pixels ($784$ features):** Converts each normalized $28 \times 28$ grayscale apparel image directly into a 1D vector of length $784$.
2. **Histogram of Oriented Gradients (HOG) ($324$ features):** Captures local edge orientations using $9$ orientation bins, $7 \times 7$ pixel cells, $2 \times 2$ cell blocks, and L2-Hys normalization:
   $$\text{Total Features} = 3 \times 3 \text{ blocks} \times 2 \times 2 \text{ cells} \times 9 \text{ orientations} = 324$$

Below, we visualize original garment images side-by-side with their HOG orientation gradient fields across all 10 Fashion-MNIST clothing categories and examine the geometric information tradeoffs.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from skimage.feature import hog

# Locate project root and ensure src is importable
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "docs/team").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks/ directory.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from aml.data import CLASS_NAMES, get_train_val_test_splits
from aml.features import extract_pixels, extract_hog_batch, extract_single_hog, get_features

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
print("Project root:", ROOT)


In [ ]:
# Load official development splits (normalized float32 in [0.0, 1.0])
data_dir = ROOT / "data" / "raw"
splits = get_train_val_test_splits(data_dir=data_dir)
X_train = splits["X_train"]
y_train = splits["y_train"]

print(f"Loaded X_train shape: {X_train.shape}, dtype: {X_train.dtype}, range: [{float(X_train.min()):.1f}, {float(X_train.max()):.1f}]")
print(f"Categories ({len(CLASS_NAMES)}): {CLASS_NAMES}")


## 1. Visualizing HOG Gradient Orientation Maps

For each of the 10 clothing categories, we select a representative sample from the training set, compute its HOG descriptors with `visualize=True`, and plot the original $28 \times 28$ image alongside its reconstructed gradient orientation field.

In [ ]:
representative_indices = [np.where(y_train == label_idx)[0][0] for label_idx in range(10)]

fig, axes = plt.subplots(10, 2, figsize=(6, 18), dpi=120)
fig.suptitle("Fashion-MNIST: Raw Images vs. HOG Gradient Maps", fontsize=13, y=0.995)

figures_dir = ROOT / "reports" / "figures" / "features"
figures_dir.mkdir(parents=True, exist_ok=True)

for row_idx, sample_idx in enumerate(representative_indices):
    label_idx = y_train[sample_idx]
    class_name = CLASS_NAMES[label_idx]
    img = X_train[sample_idx]
    
    # Compute HOG feature vector and visualization image
    _, hog_img = hog(
        img,
        orientations=9,
        pixels_per_cell=(7, 7),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        transform_sqrt=False,
        visualize=True,
        feature_vector=True,
    )
    
    # Plot original grayscale image
    axes[row_idx, 0].imshow(img, cmap="gray", vmin=0.0, vmax=1.0)
    axes[row_idx, 0].set_title(f"Class {label_idx}: {class_name}", fontsize=9, loc="left", fontweight="bold")
    axes[row_idx, 0].axis("off")
    
    # Plot HOG gradient orientation map
    axes[row_idx, 1].imshow(hog_img, cmap="inferno")
    axes[row_idx, 1].set_title("HOG Gradient Map (324-d)", fontsize=9, loc="left")
    axes[row_idx, 1].axis("off")

plt.tight_layout()
output_fig_path = figures_dir / "hog_vs_pixels.png"
plt.savefig(output_fig_path, bbox_inches="tight", dpi=150)
print(f"Saved visualization figure to: {output_fig_path}")
plt.show()


## 2. Analysis of the Geometric Information Tradeoff

Comparing **Flattened Pixels (784 dimensions)** against **HOG Descriptors (324 dimensions)** highlights core considerations in classical feature engineering for apparel classification:

### 1. Spatial Structure vs. Contrast Invariance
- **Flattened Pixels ($784$ dimensions):** Retains raw grayscale luminance values and subtle internal surface textures, but flattening images into 1D vectors destroys 2D spatial adjacency. Classifiers like Logistic Regression or Linear SVM must independently learn weights for each pixel index without intrinsic awareness of neighboring pixels.
- **HOG Descriptors ($324$ dimensions):** Encodes localized 2D edge orientations within $7 \times 7$ pixel cells and applies L2-Hys block normalization over $2 \times 2$ cells. This introduces invariance to uniform illumination shifts and highlights primary garment silhouettes (e.g., trouser legs, bag straps, boot heels).

### 2. The Clothing Ambiguity Gap (Phase 2 Motivation)
- Handcrafted HOG gradients effectively discriminate between classes with distinct structural silhouettes (such as **Trouser (1)** vs. **Bag (8)** vs. **Ankle boot (9)**).
- However, for visually similar tops—**T-shirt/top (0)**, **Pullover (2)**, **Coat (4)**, and **Shirt (6)**—outer garment silhouettes are almost identical. Crucial discriminative cues (collar geometry, button plackets, and sleeve seams) are averaged away within $7 \times 7$ cells.
- This fundamental bottleneck motivates **Phase 2**, where a Convolutional Neural Network (CNN) learns hierarchical, multi-scale visual filters directly from data rather than relying on fixed hand-engineered descriptors.

In [ ]:
# Verification of extraction dimensions and data types
test_samples = X_train[:5]

pixels_feat = extract_pixels(test_samples)
hog_feat = extract_hog_batch(test_samples)

print("--- Output Shape and Type Verification ---")
print(f"Input shape:            {test_samples.shape}, dtype: {test_samples.dtype}")
print(f"Flattened Pixels shape: {pixels_feat.shape}, dtype: {pixels_feat.dtype}")
print(f"HOG Descriptors shape:  {hog_feat.shape}, dtype: {hog_feat.dtype}")

assert pixels_feat.shape == (5, 784), f"Expected (5, 784), got {pixels_feat.shape}"
assert hog_feat.shape == (5, 324), f"Expected (5, 324), got {hog_feat.shape}"
assert pixels_feat.dtype == np.float32, f"Expected float32, got {pixels_feat.dtype}"
assert hog_feat.dtype == np.float32, f"Expected float32, got {hog_feat.dtype}"
assert np.all(np.isfinite(pixels_feat)), "Pixels contains non-finite values"
assert np.all(np.isfinite(hog_feat)), "HOG contains non-finite values"

print("All assertions passed successfully.")
